In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import itertools
import io
import math
import textwrap
import taxonomy
import time
import numpy as np
import evaluate

from PIL import Image
from pathlib import Path
from collections import Counter

/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(f'responses_judge_gpt_54.dill', 'rb') as fp:
    responses = dill.load(fp)

# LLM-as-judge

In [4]:
for paper_id, error_id, res in responses:
    if 'model_responses' in res:
        for model_response in res['model_responses']:
            try:
                judgement = model_response['judgement']['final_score']
            except:
                judgement = json.loads(model_response['judgement'])
                model_response['judgement'] = judgement

In [6]:
results = []
counts = {}
for paper_id, error_id, res in responses:
    if res['valid']:
        counts['all'] = counts.get('all', 0) + 1
        counts[error_id] = counts.get(error_id, 0) + 1

    if 'model_responses' in res:
        for model_response in res['model_responses']:
            model_id = model_response['model_id']
            #add task_id here if we add other tasks
            input_type = model_response['input_type']
            score = model_response['judgement']['final_score']

            results.append((model_id, input_type, error_id, paper_id, score))

In [28]:
# valid responses
for k,v in counts.items():
    if k == 'all':
        print(f'total: {v}')
    else:
        print(f'{base64.b64decode(k).decode():50}: {v}')

total: 270
methodology_result_conflict                       : 56
semantic_misinterpretation                        : 57
suboptimal_methodology                            : 55
contradiction_with_established_knowledge          : 47
logically_invalid_interpretation                  : 55


In [7]:
scores = {}

for model, input_type, error_id, paper_id, score in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    scores[meta_id] = scores.get(meta_id, []) + [score]

In [8]:
for meta_id, score_array in sorted(scores.items(), key=lambda x: (x[0].split('___')[0], x[0].split('___')[1], x[0].split('___')[2])):
    avg_score = np.mean(score_array)
    model, input_type, error_id = meta_id.split('___')
    error_category = base64.b64decode(error_id).decode()
    print(f'{model:15} {input_type:20} {error_category:50} {avg_score:3.2}')

gemini_25_flash images               contradiction_with_established_knowledge           1.7
gemini_25_flash images               logically_invalid_interpretation                   2.1
gemini_25_flash images               methodology_result_conflict                        1.6
gemini_25_flash images               semantic_misinterpretation                         1.8
gemini_25_flash images               suboptimal_methodology                             1.3
gemini_25_flash images_and_latex     contradiction_with_established_knowledge           2.2
gemini_25_flash images_and_latex     logically_invalid_interpretation                   4.0
gemini_25_flash images_and_latex     methodology_result_conflict                        2.8
gemini_25_flash images_and_latex     semantic_misinterpretation                         2.8
gemini_25_flash images_and_latex     suboptimal_methodology                             2.4
gemini_25_flash latex                contradiction_with_established_knowledge   

In [17]:
input_types = ['images', 'latex', 'images_and_latex']
input_types_to_name = {
    'images': 'Images',
    'latex': 'Latex',
    'images_and_latex': r'Images \& Latex',
}

error_categories = ['contradiction_with_established_knowledge', 'logically_invalid_interpretation', 'methodology_result_conflict', 'semantic_misinterpretation', 'suboptimal_methodology']
error_cat_to_acronym = {
    'contradiction_with_established_knowledge': 'ConEK',
    'logically_invalid_interpretation': 'LogII',
    'methodology_result_conflict': 'MetRC',
    'semantic_misinterpretation': 'SemM',
    'suboptimal_methodology': 'SubM'
}

error_category_names = [taxonomy.TAXONOMY_DATA[cat]['name'] for cat in error_categories]
error_category_acronyms = [error_cat_to_acronym[cat] for cat in error_categories]

model_ids = ['gpt_54', 'gpt_54_mini', 'gpt_54_nano', 'gemini_25_flash', 'gemini_25_flash_lite']
model_id_to_name = {
    'gpt_54': 'GPT 5.4', 
    'gpt_54_mini': 'GPT 5.4-mini',
    'gpt_54_nano': 'GPT 5.4-nano',
    'gemini_25_flash': 'Gemini 2.5 Flash',
    'gemini_25_flash_lite': 'Gemini 2.5 Flash-lite'
}

print(r'\begin{tabular}{c|ccccc}')
print(r'    \toprule')
#print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
print(r'    Model & ' + ' & '.join(error_category_acronyms) + r' \\')
print(r'    \midrule')

for model_id in model_ids:
    model_name = model_id_to_name[model_id]
    model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
    print(model_row)
    print(r'    \midrule')
    for input_type in input_types:
        input_name = input_types_to_name[input_type]
        outp = f'   {input_name} & '
        for error_cat in error_categories:
            error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
            score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
            score = np.mean(score_array)
            outp += f'{score:.2} & '
        print(outp[:-2] + r'\\' )

    print(r'    \midrule')
print(r'\bottomrule')
print(r'\end{tabular}')

\begin{tabular}{c|ccccc}
    \toprule
    Model & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 3.3 & 4.7 & 3.8 & 3.6 & 3.4 \\
   Latex & 3.7 & 4.8 & 4.3 & 3.5 & 3.2 \\
   Images \& Latex & 3.6 & 4.8 & 4.2 & 3.5 & 3.4 \\
    \midrule
   GPT 5.4-mini & & & & &  \\
    \midrule
   Images & 2.3 & 4.1 & 3.3 & 2.7 & 2.2 \\
   Latex & 2.9 & 4.5 & 3.7 & 2.6 & 2.1 \\
   Images \& Latex & 2.9 & 4.2 & 3.7 & 2.9 & 2.4 \\
    \midrule
   GPT 5.4-nano & & & & &  \\
    \midrule
   Images & 2.0 & 3.0 & 2.4 & 2.0 & 1.7 \\
   Latex & 2.4 & 4.3 & 3.3 & 2.4 & 1.6 \\
   Images \& Latex & 2.2 & 4.2 & 3.1 & 2.6 & 1.6 \\
    \midrule
   Gemini 2.5 Flash & & & & &  \\
    \midrule
   Images & 1.7 & 2.1 & 1.6 & 1.8 & 1.3 \\
   Latex & 2.2 & 3.9 & 2.6 & 2.9 & 2.5 \\
   Images \& Latex & 2.2 & 4.0 & 2.8 & 2.8 & 2.4 \\
    \midrule
   Gemini 2.5 Flash-lite & & & & &  \\
    \midrule
   Images & 1.7 & 2.0 & 1.9 & 1.9 & 1.5 \\
   Latex & 1.9 & 2.8 & 2.0 & 2.5

# Surface-Based Similarity metrics

These metrics are based on token overlap.

* Rouge-L: Based on the longest common subsequence. Used frequentyly in summarization.
* BLEU: Frequently used for translartion.
* BERTScore: Contextualized representations for each token, then computes similarity.

def evaluate_generated(responses, metric_name, score_key):
    metric = evaluate.load(metric_name)
    results = []
    for i, data in enumerate(responses):
        paper_id, err_id, res = data
        if i % 20 == 0:
            print(i)
            
        if 'model_responses' in res:
            ground_truth = None
            for gr in res['ground_truth']:
                if gr['task_id'] == 'generate_ground_truth_description':
                    ground_truth = gr['response_text']
                    break

            for model_response in res['model_responses']:
                model_id = model_response['model_id']
                #add task_id here if we add other tasks
                input_type = model_response['input_type']

                generated = [
                    model_response['response_text'],
                ]

                references = [[
                    ground_truth,
                ]]
                
                score = metric.compute(predictions=generated, references=references)[score_key]

                results.append((model_id, input_type, error_id, paper_id, score))

    return results

In [113]:
def evaluate_generated2(responses, metric_name, score_key, additional_kwargs):
    metric = evaluate.load(metric_name)
    generated_with_gold = {}
    for paper_id, error_id, res in responses:
        if 'model_responses' in res:
            ground_truth = None
            for gr in res['ground_truth']:
                if gr['task_id'] == 'generate_ground_truth_description':
                    ground_truth = gr['response_text']
                    break

            
            for model_response in res['model_responses']:
                model_id = model_response['model_id']
                input_type = model_response['input_type']
                meta_id = f'{model_id}___{input_type}___{error_id}'

                if meta_id not in generated_with_gold:
                    generated_with_gold[meta_id] = {
                        'generated': [],
                        'ground_truth': [],
                    }
                
                generated = model_response['response_text']

                generated_with_gold[meta_id]['generated'].append(generated)
                generated_with_gold[meta_id]['ground_truth'].append([ground_truth])

    scores = {}

    for i, item in enumerate(generated_with_gold.items()):
        if i % 20 == 0:
            print(i)

        meta_id, generated_and_gold = item
        generated = generated_and_gold['generated']
        references = generated_and_gold['ground_truth']
        score = metric.compute(predictions=generated, references=references, **additional_kwargs)[score_key]
        scores[meta_id] = scores.get(meta_id, []) + [score]

    return scores

In [103]:
def print_table(scores):
    print(r'\begin{tabular}{c|ccccc}')
    print(r'    \toprule')
    #print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
    print(r'    Model & ' + ' & '.join(error_category_acronyms) + r' \\')
    print(r'    \midrule')

    for model_id in model_ids:
        model_name = model_id_to_name[model_id]
        model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
        print(model_row)
        print(r'    \midrule')
        for input_type in input_types:
            input_name = input_types_to_name[input_type]
            outp = f'   {input_name} & '
            for error_cat in error_categories:
                error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
                score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
                score = np.mean(score_array)
                outp += f'{score:.2} & '
            print(outp[:-2] + r'\\' )

        print(r'    \midrule')
    print(r'\bottomrule')
    print(r'\end{tabular}')

In [137]:
def print_merged_table(metric1_scores, metric1_name, metric2_scores, metric2_name):
    print(r'\begin{tabular}{c|ccccc|ccccc}')
    print(r'    \toprule')
    #print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
    print(r' & \multicolumn{5}{c}{'+f'{metric1_name}'+r'} & \multicolumn{5}{c}{'+f'{metric2_name}'+r'} \\')
    print(r'    Model & ' + ' & '.join(error_category_acronyms) + ' & ' + ' & '.join(error_category_acronyms) + r' \\')
    print(r'    \midrule')

    for model_id in model_ids:
        model_name = model_id_to_name[model_id]
        model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
        print(model_row)
        print(r'    \midrule')
        for input_type in input_types:
            input_name = input_types_to_name[input_type]
            outp = f'   {input_name} & '
            for metric_scores in [metric1_scores, metric2_scores]:
                scores = metric_scores
                for error_cat in error_categories:
                    error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
                    score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
                    score = np.mean(score_array)
                    outp += f'{score:.2f} & '
            print(outp[:-2] + r'\\' )

        print(r'    \midrule')
    print(r'\bottomrule')
    print(r'\end{tabular}')

## Rouge

In [ ]:
rouge = evaluate_generated2(responses, "rouge", "rougeL", {"use_aggregator": False})

0
20
40
60


In [ ]:
print_table(rouge)

\begin{tabular}{c|ccccc}
    \toprule
    Model & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.11 & 0.15 & 0.12 & 0.12 & 0.12 \\
   Latex & 0.11 & 0.13 & 0.12 & 0.12 & 0.12 \\
   Images \& Latex & 0.11 & 0.13 & 0.11 & 0.12 & 0.12 \\
    \midrule
   GPT 5.4-mini & & & & &  \\
    \midrule
   Images & 0.13 & 0.19 & 0.15 & 0.15 & 0.14 \\
   Latex & 0.14 & 0.2 & 0.16 & 0.15 & 0.14 \\
   Images \& Latex & 0.13 & 0.17 & 0.15 & 0.14 & 0.13 \\
    \midrule
   GPT 5.4-nano & & & & &  \\
    \midrule
   Images & 0.12 & 0.17 & 0.13 & 0.13 & 0.13 \\
   Latex & 0.13 & 0.18 & 0.14 & 0.13 & 0.12 \\
   Images \& Latex & 0.12 & 0.19 & 0.14 & 0.13 & 0.12 \\
    \midrule
   Gemini 2.5 Flash & & & & &  \\
    \midrule
   Images & 0.087 & 0.086 & 0.088 & 0.091 & 0.098 \\
   Latex & 0.11 & 0.13 & 0.1 & 0.12 & 0.12 \\
   Images \& Latex & 0.096 & 0.13 & 0.11 & 0.12 & 0.11 \\
    \midrule
   Gemini 2.5 Flash-lite & & & & &  \\
    \midrule
   Images &

## BERTScore

In [114]:
bertscore = evaluate_generated2(responses, "bertscore", "recall", {"lang": "en"})

0


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


20


40
60


In [138]:
print_merged_table(rouge2, 'Rouge-L', bertscore, 'BERTScore (Recall)')

\begin{tabular}{c|ccccc|ccccc}
    \toprule
 & \multicolumn{5}{c}{Rouge-L} & \multicolumn{5}{c}{BERTScore (Recall)} \\
    Model & ConEK & LogII & MetRC & SemM & SubM & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.11 & 0.15 & 0.12 & 0.12 & 0.12 & 0.83 & 0.86 & 0.84 & 0.81 & 0.82 \\
   Latex & 0.11 & 0.13 & 0.12 & 0.12 & 0.12 & 0.84 & 0.86 & 0.85 & 0.83 & 0.83 \\
   Images \& Latex & 0.11 & 0.13 & 0.11 & 0.12 & 0.12 & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-mini & & & & &  \\
    \midrule
   Images & 0.13 & 0.19 & 0.15 & 0.15 & 0.14 & 0.83 & 0.86 & 0.84 & 0.83 & 0.83 \\
   Latex & 0.14 & 0.20 & 0.16 & 0.15 & 0.14 & 0.83 & 0.87 & 0.85 & 0.84 & 0.83 \\
   Images \& Latex & 0.13 & 0.17 & 0.15 & 0.14 & 0.13 & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-nano & & & & &  \\
    \midrule
   Images & 0.12 & 0.17 & 0.13 & 0.13 & 0.13 & 0.83 & 0.85 & 0.83 & 0.82 & 0.82 \\
   Latex & 0.13 & 0.18 & 0.14 & 

# Human-Validated Subset

In [140]:
def get_choice_value(annotation):
    annotations = annotation['annotations']
    for ann in annotations:
        if ann['was_cancelled']:
            return None
        else:
            for res in ann['result']:
                if res['type'] == 'choices' and res['from_name'] == 'edit_correct':
                    choice = res['value']['choices'][0]
                    if choice == 'yes':
                        return 1
                    else:
                        return 0

In [204]:
def print_merged_table(metric1_scores, metric1_name, metric2_scores, metric2_name, precision=1):
    print(r'\begin{tabular}{c|ccccc|ccccc}')
    print(r'    \toprule')
    #print(r'    Model & \multicolumn{5}{|c|}{images} & \multicolumn{5}{|c|}{latex} & \multicolumn{5}{|c|}{images_and_latex} \\')
    print(r' & \multicolumn{5}{c}{'+f'{metric1_name}'+r'} & \multicolumn{5}{c}{'+f'{metric2_name}'+r'} \\')
    print(r'    Model & ' + ' & '.join(error_category_acronyms) + ' & ' + ' & '.join(error_category_acronyms) + r' \\')
    print(r'    \midrule')

    for model_id in model_ids:
        model_name = model_id_to_name[model_id]
        model_row = f'   {model_name} & ' + '& ' * (len(error_category_acronyms)-1) + r' \\'
        print(model_row)
        print(r'    \midrule')
        for input_type in input_types:
            input_name = input_types_to_name[input_type]
            outp = f'   {input_name} & '
            for metric_scores in [metric1_scores, metric2_scores]:
                scores = metric_scores
                for error_cat in error_categories:
                    error_cat_b64 = base64.urlsafe_b64encode(error_cat.encode()).decode()
                    score_array = scores[f'{model_id}___{input_type}___{error_cat_b64}']
                    score = np.mean(score_array)
                    if precision == 1:
                        outp += f'{score:.1f} & '
                    else:
                        outp += f'{score:.2f} & '
            print(outp[:-2] + r'\\' )

        print(r'    \midrule')
    print(r'\bottomrule')
    print(r'\end{tabular}')

In [163]:
HUMAN_ANNOTATIONS_FILE_PATH = Path('./aj_main.json')
with open(HUMAN_ANNOTATIONS_FILE_PATH, 'r') as fp:
    annotations = json.load(fp)

In [164]:
human_validation = {}
for annotation in annotations:
    paper_id = annotation['data']['id']
    error_id = annotation['data']['error_id']
    meta_id = f'{paper_id}___{error_id}'
    annotation_value = get_choice_value(annotation)
    error_name = base64.urlsafe_b64decode(error_id).decode()
    human_validation[meta_id] = annotation_value

In [165]:
with open(f'responses_judge_gpt_54.dill', 'rb') as fp:
    responses = dill.load(fp)

for paper_id, error_id, res in responses:
    if 'model_responses' in res:
        for model_response in res['model_responses']:
            try:
                judgement = model_response['judgement']['final_score']
            except:
                judgement = json.loads(model_response['judgement'])
                model_response['judgement'] = judgement

results = []
for paper_id, error_id, res in responses:
    if 'model_responses' in res:
        for model_response in res['model_responses']:
            model_id = model_response['model_id']
            #add task_id here if we add other tasks
            input_type = model_response['input_type']
            score = model_response['judgement']['final_score']
            human_valid = human_validation.get(f'{paper_id}___{error_id}', None)
            results.append((model_id, input_type, error_id, paper_id, score, human_valid))

In [166]:
scores = {}

for model, input_type, error_id, paper_id, score, human_valid in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    scores[meta_id] = scores.get(meta_id, []) + [score]

In [167]:
human_valid_scores = {}

for model, input_type, error_id, paper_id, score, human_valid in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    if human_valid == 1:
        human_valid_scores[meta_id] = human_valid_scores.get(meta_id, []) + [score]

In [172]:
assert len(scores) == len(human_valid_scores.keys()) # model_input_errorid should be same

In [179]:
print_merged_table(scores, 'Full Dataset', human_valid_scores, 'Human Validated Subset')

\begin{tabular}{c|ccccc|ccccc}
    \toprule
 & \multicolumn{5}{c}{Full Dataset} & \multicolumn{5}{c}{Human Validated Subset} \\
    Model & ConEK & LogII & MetRC & SemM & SubM & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 3.3 & 4.7 & 3.8 & 3.6 & 3.4 & 3.6 & 4.5 & 3.7 & 3.6 & 3.8 \\
   Latex & 3.7 & 4.8 & 4.3 & 3.5 & 3.2 & 4.0 & 4.7 & 4.1 & 3.8 & 3.4 \\
   Images \& Latex & 3.6 & 4.8 & 4.2 & 3.5 & 3.4 & 3.8 & 4.7 & 3.9 & 3.6 & 3.5 \\
    \midrule
   GPT 5.4-mini & & & & &  \\
    \midrule
   Images & 2.3 & 4.1 & 3.3 & 2.7 & 2.2 & 2.4 & 3.6 & 3.1 & 2.8 & 2.4 \\
   Latex & 2.9 & 4.5 & 3.7 & 2.6 & 2.1 & 2.8 & 4.4 & 3.6 & 2.6 & 2.0 \\
   Images \& Latex & 2.9 & 4.2 & 3.7 & 2.9 & 2.4 & 3.0 & 3.9 & 3.6 & 3.1 & 2.8 \\
    \midrule
   GPT 5.4-nano & & & & &  \\
    \midrule
   Images & 2.0 & 3.0 & 2.4 & 2.0 & 1.7 & 2.1 & 3.0 & 2.3 & 2.1 & 1.4 \\
   Latex & 2.4 & 4.3 & 3.3 & 2.4 & 1.6 & 2.6 & 4.3 & 3.2 & 2.6 & 1.4 \\
   Images \& Latex & 

In [183]:
rouge_score_human_valid = {}

#paper ids should be in same order]
filter_mask = {}
for model, input_type, error_id, paper_id, score, human_valid in results:
    meta_id = f'{model}___{input_type}___{error_id}'
    filter_mask[meta_id] = filter_mask.get(meta_id, []) + [human_valid == 1]

In [201]:
for k, v in filter_mask.items():
    assert len(v) == len(bertscore[k][0]) == len(rouge2[k][0])

In [202]:
bertscore_human_validated = {}
rouge_human_validated = {}
for k, v in filter_mask.items():
    bertscore_human_validated[k] = [s for s, valid in zip(bertscore[k][0], v) if valid]
    rouge_human_validated[k] = [s for s, valid in zip(rouge2[k][0], v) if valid]

In [205]:
print_merged_table(rouge_human_validated, 'Rouge-L', bertscore_human_validated, 'BERTScore (Recall)', precision=2)

\begin{tabular}{c|ccccc|ccccc}
    \toprule
 & \multicolumn{5}{c}{Rouge-L} & \multicolumn{5}{c}{BERTScore (Recall)} \\
    Model & ConEK & LogII & MetRC & SemM & SubM & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.11 & 0.15 & 0.11 & 0.13 & 0.12 & 0.83 & 0.85 & 0.84 & 0.83 & 0.82 \\
   Latex & 0.11 & 0.14 & 0.12 & 0.12 & 0.12 & 0.84 & 0.86 & 0.85 & 0.83 & 0.83 \\
   Images \& Latex & 0.10 & 0.13 & 0.11 & 0.12 & 0.12 & 0.84 & 0.85 & 0.84 & 0.83 & 0.83 \\
    \midrule
   GPT 5.4-mini & & & & &  \\
    \midrule
   Images & 0.12 & 0.19 & 0.14 & 0.15 & 0.13 & 0.83 & 0.85 & 0.83 & 0.83 & 0.82 \\
   Latex & 0.13 & 0.20 & 0.15 & 0.15 & 0.13 & 0.83 & 0.87 & 0.85 & 0.84 & 0.82 \\
   Images \& Latex & 0.13 & 0.17 & 0.14 & 0.14 & 0.13 & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-nano & & & & &  \\
    \midrule
   Images & 0.12 & 0.17 & 0.13 & 0.13 & 0.12 & 0.83 & 0.85 & 0.83 & 0.82 & 0.82 \\
   Latex & 0.12 & 0.18 & 0.13 & 

In [207]:
print_table(bertscore)

\begin{tabular}{c|ccccc}
    \toprule
    Model & ConEK & LogII & MetRC & SemM & SubM \\
    \midrule
   GPT 5.4 & & & & &  \\
    \midrule
   Images & 0.83 & 0.86 & 0.84 & 0.81 & 0.82 \\
   Latex & 0.84 & 0.86 & 0.85 & 0.83 & 0.83 \\
   Images \& Latex & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-mini & & & & &  \\
    \midrule
   Images & 0.83 & 0.86 & 0.84 & 0.83 & 0.83 \\
   Latex & 0.83 & 0.87 & 0.85 & 0.84 & 0.83 \\
   Images \& Latex & 0.83 & 0.86 & 0.84 & 0.83 & 0.82 \\
    \midrule
   GPT 5.4-nano & & & & &  \\
    \midrule
   Images & 0.83 & 0.85 & 0.83 & 0.82 & 0.82 \\
   Latex & 0.84 & 0.87 & 0.85 & 0.84 & 0.83 \\
   Images \& Latex & 0.84 & 0.87 & 0.84 & 0.84 & 0.82 \\
    \midrule
   Gemini 2.5 Flash & & & & &  \\
    \midrule
   Images & 0.81 & 0.82 & 0.82 & 0.81 & 0.81 \\
   Latex & 0.83 & 0.84 & 0.83 & 0.83 & 0.83 \\
   Images \& Latex & 0.83 & 0.84 & 0.83 & 0.83 & 0.82 \\
    \midrule
   Gemini 2.5 Flash-lite & & & & &  \\
    \midrule
   Images & 0.8